# Chapter 32 – Worked examples

This notebook runs the examples of [Chapter 32](../index.md) that do not need ToOp itself: the identity-hash
trap, the traced loop bound, platform integer widths and buffer donation. The code lives in the modules next to
it (`stale_hash.py`, `traced_loop.py`, `int_widths.py`, `pmap_donate.py`); the cells only call them. The
measurements on ToOp's solver (`solver_lab_toop.py`, `split_slots_toop.py`) run in the ToOp container: see
[lab_toop.ipynb](lab_toop.ipynb) and [solutions.md](../solutions.md).

Run it from this folder (the kernel's working directory must be `src/`).

In [ ]:
import int_widths
import jax
import jax.numpy as jnp
import pmap_donate
import stale_hash
import traced_loop

print('JAX', jax.__version__, 'on', jax.default_backend(), '- x64 enabled:', jax.config.jax_enable_x64)

## The stale-result trap

`IdentityConfig` hashes by `id(self)`, like ToOp's `SolverConfig`. A field mutated in place leaves the hash
unchanged, so the jitted function returns the result compiled for the *old* value; `dataclasses.replace` makes a
new object, a new cache key, and recompiles. `FrozenConfig` hashes by value and does the right thing by itself.

In [ ]:
flows = jnp.array([10.0, -80.0, 30.0, -5.0, 60.0, 20.0, -45.0, 70.0])
print('IdentityConfig  (first, mutated in place, replace):', stale_hash.trap(flows))
print('FrozenConfig    (3, 5, traces):', stale_hash.safe(flows))

## A traced loop bound

`lax.fori_loop` with a traced upper bound compiles once for every Poisson draw; a Python `for` over the same
draw must take it as a static integer and compiles once per distinct value. The jaxprs show why: one `while`
against an unrolled body.

In [ ]:
table = jnp.arange(1.0, 2 * traced_loop.MAX_STEPS + 1.0)
draws = [int(traced_loop.draw_steps(k)) for k in jax.random.split(jax.random.key(0), 20)]
print('draws:', draws)
print('traces (fori_loop, Python for):', traced_loop.run_both(table, draws), '- distinct draws:', len(set(draws)))
print(jax.make_jaxpr(traced_loop._apply_fori)(table, jnp.asarray(3), 0))

## What `int` means on this machine

NumPy 2 makes `dtype(int)` 64-bit on every platform, while `'long'` still follows the C `long` (32-bit on
Windows). JAX's `dtype=int` follows `jax_enable_x64`, which is read when JAX starts, so the second report runs
in a fresh interpreter.

In [ ]:
for key, value in int_widths.numpy_report().items():
    print(f'{key:22s} {value}')
print('x64 on :', int_widths.jax_report_in_subprocess(True))
print('x64 off:', int_widths.jax_report_in_subprocess(False))

## Four CPU devices and buffer donation

`XLA_FLAGS=--xla_force_host_platform_device_count=4` must be set before JAX starts, so the demonstration runs
in a subprocess. On CPU the donated `pmap` input stays readable; on a GPU it is deleted.

In [ ]:
print(pmap_donate.run_in_subprocess())